# 24 — Event Correlation Timeline

April 2025 replacement.

In [ ]:

DATE_FROM = "2025-04-02"
DATE_TO   = "2025-04-07"
OUTPUT_DIR = "outputs/event_correlation"
GRADIENT_CSV = "outputs/gradient/coastal_gradient_daily_summary.csv"
CLEAR_SKY_CSV = "outputs/clear_sky_snapshot/clear_sky_shortlist.csv"
WIND_CALM_CSV = "outputs/wind_calm_test/wind_calm_candidate_days.csv"
BACK_TRAJ_CSV = "outputs/back_trajectory/NHV_back_trajectory.csv"
PLACE_TERMS = ["Newhaven","Lewes","Eastbourne","Seaford","Peacehaven","Shoreham"]
EVENT_TERMS = ["fire","smoke","industrial","plant","incinerator","pollution","traffic","port","gas leak"]
MAX_ARTICLES_PER_SOURCE = 30


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

gradient,g_meta=safe_read_csv(GRADIENT_CSV); clear_sky,c_meta=safe_read_csv(CLEAR_SKY_CSV); wind_calm,w_meta=safe_read_csv(WIND_CALM_CSV); back_traj,b_meta=safe_read_csv(BACK_TRAJ_CSV)
input_files=[g_meta,c_meta,w_meta,b_meta]
timeline_rows=[]
def add_rows(df,label):
    if "date" not in df.columns: return
    for _,r in df.iterrows():
        timeline_rows.append({"date":str(r.get("date")),"source":label,"site_id":r.get("site_id"),"site_name":r.get("site_name"),"signal_strength":r.get("scene_count",r.get("scenes",r.get("mean_wind_speed_ms",np.nan)))})
add_rows(gradient,"gradient"); add_rows(clear_sky,"clear_sky"); add_rows(wind_calm,"wind_calm")
timeline=pd.DataFrame(timeline_rows)
if not timeline.empty: timeline["date"]=pd.to_datetime(timeline["date"],errors="coerce").dt.date.astype("string")
NEWSAPI_KEY=os.getenv("NEWSAPI_KEY"); NEWSDATA_KEY=os.getenv("NEWSDATA_KEY") or os.getenv("NEWSDATA_IO_KEY")
QUERY_STRING=" OR ".join(PLACE_TERMS)+" AND ("+" OR ".join(EVENT_TERMS)+")"
def request_json_with_retry(url,params=None,retries=3,timeout=40):
    last_err=None
    for attempt in range(retries):
        try:
            r=requests.get(url,params=params,timeout=timeout); r.raise_for_status(); return r.json()
        except Exception as e:
            last_err=e
            if attempt<retries-1: time.sleep(2**attempt)
    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")
article_rows=[]; api_log=[]
if NEWSAPI_KEY:
    try:
        data=request_json_with_retry("https://newsapi.org/v2/everything",params={"q":QUERY_STRING,"from":DATE_FROM,"to":DATE_TO,"language":"en","pageSize":MAX_ARTICLES_PER_SOURCE,"apiKey":NEWSAPI_KEY})
        for a in data.get("articles",[]):
            article_rows.append({"source_api":"newsapi","published_utc":a.get("publishedAt"),"title":a.get("title"),"description":a.get("description"),"source_name":(a.get("source") or {}).get("name"),"url":a.get("url"),"content":a.get("content")})
        api_log.append({"api":"newsapi","status":"ok","rows":len(data.get("articles",[]))})
    except Exception as e: api_log.append({"api":"newsapi","status":"error","error":str(e)})
else: api_log.append({"api":"newsapi","status":"missing_key"})
if NEWSDATA_KEY:
    try:
        data=request_json_with_retry("https://newsdata.io/api/1/news",params={"apikey":NEWSDATA_KEY,"q":QUERY_STRING,"country":"gb","language":"en"})
        for a in data.get("results",[])[:MAX_ARTICLES_PER_SOURCE]:
            article_rows.append({"source_api":"newsdata","published_utc":a.get("pubDate"),"title":a.get("title"),"description":a.get("description") or a.get("content"),"source_name":a.get("source_id") or a.get("source_name"),"url":a.get("link"),"content":a.get("content")})
        api_log.append({"api":"newsdata","status":"ok","rows":min(len(data.get("results",[])),MAX_ARTICLES_PER_SOURCE)})
    except Exception as e: api_log.append({"api":"newsdata","status":"error","error":str(e)})
else: api_log.append({"api":"newsdata","status":"missing_key"})
articles=pd.DataFrame(article_rows)
if not articles.empty: articles["published_utc"]=pd.to_datetime(articles["published_utc"],utc=True,errors="coerce"); articles["date"]=articles["published_utc"].dt.date.astype("string")
else: articles=pd.DataFrame(columns=["source_api","published_utc","title","description","source_name","url","content","date"])
article_day=articles.groupby("date",dropna=False).agg(article_count=("url","count"),top_title=("title","first")).reset_index() if not articles.empty else pd.DataFrame(columns=["date","article_count","top_title"])
summary=timeline.merge(article_day,on="date",how="left") if not timeline.empty else article_day.copy()
if "article_count" not in summary.columns: summary["article_count"]=0
summary["article_count"]=pd.to_numeric(summary["article_count"],errors="coerce").fillna(0); summary["forensic_interest_score"]=pd.to_numeric(summary.get("signal_strength",0),errors="coerce").fillna(0)+summary["article_count"]
articles.to_csv(Path(OUTPUT_DIR)/"event_articles.csv",index=False); timeline.to_csv(Path(OUTPUT_DIR)/"event_timeline.csv",index=False); summary.to_csv(Path(OUTPUT_DIR)/"event_summary_by_day.csv",index=False)
display(summary.head(30))


In [ ]:

fig,ax=plt.subplots(figsize=(10,5))
if not summary.empty and "date" in summary.columns:
    s=summary.groupby("date",dropna=False).agg(article_count=("article_count","max"),forensic_interest_score=("forensic_interest_score","max")).reset_index()
    x=pd.to_datetime(s["date"]); ax.plot(x,s["article_count"],marker="o",label="article_count"); ax.plot(x,s["forensic_interest_score"],marker="o",label="forensic_interest_score")
ax.set_title("Event correlation timeline"); ax.set_xlabel("Date"); ax.set_ylabel("Score")
if not summary.empty: ax.legend()
fig.autofmt_xdate(); fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/"event_correlation_plot.png"; fig.savefig(plot_path,dpi=150); plt.show()
(Path(OUTPUT_DIR)/"event_correlation_manifest.json").write_text(json.dumps({"created_utc":datetime.now(timezone.utc).isoformat(),"input_files":input_files,"api_log":api_log,"timeline_rows":int(len(timeline)),"articles_rows":int(len(articles)),"summary_rows":int(len(summary))},indent=2),encoding="utf-8")
print("Saved:",plot_path)
